# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

*Important*: All dataset entities are referenced by their `@id` fields. Let's enumerate the available record sets to see how to access them for downstream extraction and exploration.

In [ ]:
# List all available record set @ids with their details
record_sets = metadata.record_sets
print(f"# Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '(No description)'}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}) [dataType={field.data_type if hasattr(field, 'data_type') else 'unknown'}]")
    print()

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames for analysis. Each record set and field is referenced using the unique `@id` value as shown above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load all record sets into dataframes for exploration
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded record set: {record_set_id} with shape {df.shape}")
        else:
            print(f"Record set {record_set_id} yielded 0 records.")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Pick the main protein table for demonstration (choose the one with the largest number of columns/records)
if dataframes:
    largest_rs = max(dataframes, key=lambda x: dataframes[x].shape[1])
    print(f"\nExample dataframe columns for RecordSet {largest_rs}:")
    print(dataframes[largest_rs].columns.tolist())
    display(dataframes[largest_rs].head())
else:
    print("No dataframes loaded. Verify dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All references to fields/columns use their `@id`.

In [ ]:
# Choose the main protein table for analysis
main_record_set_id = largest_rs
df = dataframes[main_record_set_id]

# Show numeric columns for selection
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric columns available: {numeric_columns}")
# If possible, choose `cr:MW_kDa` (molecular weight, as an example) or pick the first numeric column
example_numeric_field = 'cr:MW_kDa' if 'cr:MW_kDa' in numeric_columns else (numeric_columns[0] if numeric_columns else None)
if example_numeric_field is None:
    print("No numeric field found for EDA.")
else:
    # Filter: Show entries with MW > threshold
    threshold = 70
    filtered_df = df[df[example_numeric_field] > threshold]
    print(f"Filtered records with {example_numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{example_numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) / filtered_df[example_numeric_field].std()
    print(f"\nSample normalized values of '{example_numeric_field}':")
    display(filtered_df[[example_numeric_field, norm_col]].head())

    # Attempt to group by a field such as protein description if available
    group_field = 'cr:description' if 'cr:description' in df.columns else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[example_numeric_field].mean().reset_index()
        print(f"\nGrouped mean MW by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between numeric fields. Example: distribution of molecular weights.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if example_numeric_field is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[example_numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(f"{example_numeric_field} (MW in kDa)")
    plt.title(f"Distribution of {example_numeric_field} in record set {main_record_set_id}")
    plt.show()

    # Boxplot by protein description if available
    if group_field:
        # Limit to top proteins by count for readability
        top_proteins = df[group_field].value_counts().nlargest(5).index
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=example_numeric_field, data=df[df[group_field].isin(top_proteins)])
        plt.title(f"{example_numeric_field} grouped by '{group_field}' (top 5 proteins)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² mass spectrometry dataset using the `mlcroissant` library.

- **Record sets and fields** were accessed using their `@id` attributes for robust referencing.
- Loaded main data tables into dataframes; filtered and normalized protein attributes.
- Visualized protein characteristic distributions.

The `mlcroissant` library makes it easy to programmatically explore standardized datasets for bioinformatics, FAIR data, and reproducible science.